In [ ]:
# import libraries

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as pl
from matplotlib.lines import Line2D

import os

Load files

In [2]:
# Load total pop
total_pop_folder = "Population Data"
pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_age_breakdown.parquet"))

# Load student pop
student_pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_stu_breakdown.parquet"))

# Load school pop
school_folder = "School Data"
school_gdf = gpd.read_parquet(os.path.join(school_folder, "swk_list_of_schools_2025.parquet"))

# Load combined routes
routes_folder = "OSRM Routes to Nearest School"
secondary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "secondary_combined_routes.parquet"))
primary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "primary_combined_routes.parquet"))

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

In [3]:
# Create total, primary and secondary population gdfs
total_pop_gdf = pop_gdf.copy()[["id","total_pop","x","y","district","geometry"]]
primary_pop_gdf = student_pop_gdf.copy()[["id","primary_school_students","x","y","district","geometry"]]
secondary_pop_gdf = student_pop_gdf.copy()[["id","secondary_school_students","x","y","district","geometry"]]

# Round numbders
total_pop_gdf["total_pop"] = total_pop_gdf["total_pop"].round(0).astype(int)
primary_pop_gdf["primary_school_students"] = primary_pop_gdf["primary_school_students"].round(0).astype(int)
secondary_pop_gdf["secondary_school_students"] = secondary_pop_gdf["secondary_school_students"].round(0).astype(int)

# Rename columns
total_pop_gdf = total_pop_gdf.rename(columns={"id":"pop_id","total_pop":"total_population"})
primary_pop_gdf = primary_pop_gdf.rename(columns={"id":"pop_id"})
secondary_pop_gdf = secondary_pop_gdf.rename(columns={"id":"pop_id"})

In [4]:
# Create primary and school gdf
primary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Primary"][["id","nama_sekolah","bil_murid","bil_guru","district","geometry"]]
secondary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Secondary"][["id","nama_sekolah","bil_murid","bil_guru","district","geometry"]]

# Rename columns
primary_school_gdf = primary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})
secondary_school_gdf = secondary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})

Plot maps by district

In [5]:
# Folder to save outputs
out_dir = "District Charts/School_Population_Maps"
os.makedirs(out_dir, exist_ok=True)

# List of districts (adjust column name if needed)
district_list = sorted(swk_districts_gdf["name"].unique())

for district_filter in district_list:
    print(f"Plotting {district_filter}...")

    # --- Filter data ---
    total_pop_gdf_f       = total_pop_gdf[total_pop_gdf["district"] == district_filter]
    primary_school_gdf_f  = primary_school_gdf[primary_school_gdf["district"] == district_filter]
    secondary_school_gdf_f = secondary_school_gdf[secondary_school_gdf["district"] == district_filter]

    district_f = swk_districts_gdf[swk_districts_gdf["name"] == district_filter]

    # Skip if no geometry (just in case)
    if district_f.empty:
        continue

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
    ax_primary, ax_secondary = axes

    # Extent from district
    minx, miny, maxx, maxy = district_f.total_bounds

    # ------------------------
    # Left: Primary schools
    # ------------------------
    district_f.boundary.plot(ax=ax_primary, linewidth=1, edgecolor="grey")

    total_pop_gdf_f.plot(
        ax=ax_primary,
        marker=".",
        markersize=3,
        color="black",
        alpha=0.7,
        label="Population"
    )

    if not primary_school_gdf_f.empty:
        primary_school_gdf_f.plot(
            ax=ax_primary,
            marker="^",
            markersize=50,
            edgecolor="black",
            facecolor="tab:blue",
            label="Primary school"
        )

    ax_primary.set_title(f"{district_filter} – Primary Schools")
    ax_primary.set_xlim(minx, maxx)
    ax_primary.set_ylim(miny, maxy)
    ax_primary.set_xlabel("")
    ax_primary.set_ylabel("")

    # ------------------------
    # Right: Secondary schools
    # ------------------------
    district_f.boundary.plot(ax=ax_secondary, linewidth=1, edgecolor="grey")

    total_pop_gdf_f.plot(
        ax=ax_secondary,
        marker=".",
        markersize=3,
        color="black",
        alpha=0.7,
        label="Population"
    )

    if not secondary_school_gdf_f.empty:
        secondary_school_gdf_f.plot(
            ax=ax_secondary,
            marker="s",
            markersize=50,
            edgecolor="black",
            facecolor="tab:orange",
            label="Secondary school"
        )

    ax_secondary.set_title(f"{district_filter} – Secondary Schools")
    ax_secondary.set_xlabel("")

    # ------------------------
    # Shared formatting
    # ------------------------
    for ax in axes:
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(axis="both", which="both", length=0,
                       labelbottom=False, labelleft=False)

    # Legend (same for all)
    legend_elements = [
        Line2D([0], [0], marker=".", linestyle="None", color="black",
               markersize=6, label="Population points"),
        Line2D([0], [0], marker="^", linestyle="None",
               markerfacecolor="tab:blue", markeredgecolor="black",
               markersize=8, label="Primary school"),
        Line2D([0], [0], marker="s", linestyle="None",
               markerfacecolor="tab:orange", markeredgecolor="black",
               markersize=8, label="Secondary school"),
    ]
    ax_secondary.legend(handles=legend_elements, loc="best", frameon=True)

    plt.tight_layout(pad=1, w_pad=0.5, h_pad=0.5)

    # --- Save and close ---
    safe_name = district_filter.replace(" ", "_")
    out_path = os.path.join(out_dir, f"{safe_name}_schools_pop.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


Plotting Asajaya...
Plotting Bau...
Plotting Belaga...
Plotting Beluru...
Plotting Betong...
Plotting Bintulu...
Plotting Bukit Mabong...
Plotting Dalat...
Plotting Daro...
Plotting Julau...
Plotting Kabong...
Plotting Kanowit...
Plotting Kapit...
Plotting Kuching...
Plotting Lawas...
Plotting Limbang...
Plotting Lubok Antu...
Plotting Lundu...
Plotting Marudi...
Plotting Matu...
Plotting Meradong...
Plotting Miri...
Plotting Mukah...
Plotting Pakan...
Plotting Pusa...
Plotting Samarahan...
Plotting Saratok...
Plotting Sarikei...
Plotting Sebauh...
Plotting Selangau...
Plotting Serian...
Plotting Sibu...
Plotting Simunjan...
Plotting Song...
Plotting Sri Aman...
Plotting Subis...
Plotting Tanjung Manis...
Plotting Tatau...
Plotting Tebedu...
Plotting Telang Usan...
